In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from math import log, sqrt, exp
from datetime import datetime, timedelta
import os

def black_scholes_call(S, K, T, r, sigma):
    d1 = (log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    call_price = S * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)
    return call_price

def implied_volatility(S, K, T, r, C):
    def f(sigma):
        return black_scholes_call(S, K, T, r, sigma) - C

    return brentq(f, 0.001, 100, xtol=1e-6, rtol=1e-6, maxiter=100)

def compute_vix_index(prices, call_strikes, options_data, current_date, stock_symbol):
    # Extract the relevant data for the current date and stock symbol
    call_data = options_data[current_date][stock_symbol]
    
    # Filter the call options data
    call_options = call_data[call_data['type'] == 'call']
    
    # Calculate the average call strike price and days to expiration
    call_strike_avg = np.mean(call_options['strike'])
    days_to_expiration_avg = np.mean(call_options['expiration'] - datetime.strptime(current_date, '%Y-%m-%d').date()).days
    
    # Calculate the call option prices using Black-Scholes formula
    sigma = implied_volatility(prices, call_strike_avg, days_to_expiration_avg / 365, r, 0)
    call_prices = black_scholes_call(prices, call_strike_avg, days_to_expiration_avg / 365, r, sigma)
    
    # Calculate the VIX index
    vix_index = 100 * np.sqrt(days_to_expiration_avg) * np.mean(call_prices / K)
    
    return vix_index



# Specify the directory and date range
directory = '/Users/Farid Srouji/Documents/Research/master/Data/2013-01/'
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2013-01-31')

target_stock = 'A'  # Specify the stock symbol for which you want to calculate the VIX index

vix_data = []
# Iterate over the date range
while start_date <= end_date:
    # Format the current date
    current_date = start_date.strftime('%Y-%m-%d')
    
    try:
        # Construct the file paths for options and stock data
        option_file_path = os.path.join(directory, start_date.strftime("%Y-%m-%d") + "options.cvs")
        stock_file_path = os.path.join(directory, start_date.strftime("%Y-%m-%d") + "stocks.cvs")
    except:
        print("Input Files do not exist:"+current_date)
    # Check if both options and stock data files exist for the current date
    if os.path.exists(option_file_path) and os.path.exists(stock_file_path):
        # Read options and stock data into DataFrames
        options_data = pd.read_csv(option_file_path)
        stocks_data = pd.read_csv(stock_file_path)

        # Get the stock symbol for the current date
        stock_symbol = stocks_data['symbol'].values[0]
        
        # Filter options data for the target stock
        options_data = options_data[options_data['underlying'] == stock_symbol]
        
        # Extract the call strikes and expiration dates
        call_strikes = options_data[options_data['type'] == 'call']['strike'].values
        
        # Calculate the underlying stock price
        prices = stocks_data['close'].values[0]
        
        # Calculate the number of days to expiration for each option contract
        expiration_dates = pd.to_datetime(options_data['expiration'])
        days_to_expiration = (expiration_dates - start_date).dt.days
        
        # Calculate the VIX index for the current date
        vix_index = compute_vix_index(prices, call_strikes, options_data, start_date, target_stock)
        
        # Append the VIX index to the list
        vix_data.append({
            'Date': start_date,
            'Stock': target_stock,
            'VIX': vix_index
        })
    
    print(start_date)
    # Move to the next date
    print(type(start_date))
    start_date += datetime.timedelta(days=1)

# Create a DataFrame from the VIX data
vix_df = pd.DataFrame(vix_data)

# Print the VIX DataFrame
print(vix_df)

In [ ]:
# Specify the directory and date range
directory = '/Users/Farid Srouji/Documents/Research/master/Data/2013-01/'
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2013-01-31')

target_stock = 'A'  # Specify the stock symbol for which you want to calculate the VIX index

vix_index_list = []

# Iterate over the date range
while start_date <= end_date:
    # Format the current date
    current_date = start_date.strftime('%Y-%m-%d')

    # Construct the file paths for options and stock data
    option_file_path = os.path.join(directory, start_date.strftime("%Y-%m-%d") + "options.cvs")
    stock_file_path = os.path.join(directory, start_date.strftime("%Y-%m-%d") + "stocks.cvs")

    # Load the stock data
    stock_data = pd.read_csv(stock_file_path)

    # Extract the closing prices
    prices = stock_data['Close'].values

    # Load the options data
    options_data = {}
    options_data_df = pd.read_csv(option_file_path, parse_dates=['quote_date'])
    for _, row in options_data_df.iterrows():
        contract = row['contract']
        underlying = row['underlying']
        expiration = row['expiration']
        option_type = row['type']
        strike = row['strike']
        style = row['style']
        bid = row['bid']
        bid_size = row['bid_size']
        ask = row['ask']
        ask_size = row['ask_size']
        volume = row['volume']
        open_interest = row['open_interest']
        quote_date = row['quote_date']
        delta = row['delta']
        gamma = row['gamma']
        theta = row['theta']
        vega = row['vega']
        implied_volatility = row['implied_volatility']
        
        option = {
            'contract': contract,
            'underlying': underlying,
            'expiration': expiration,
            'type': option_type,
            'strike': strike,
            'style': style,
            'bid': bid,
            'bid_size': bid_size,
            'ask': ask,
            'ask_size': ask_size,
            'volume': volume,
            'open_interest': open_interest,
            'quote_date': quote_date,
            'delta': delta,
            'gamma': gamma,
            'theta': theta,
            'vega': vega,
            'implied_volatility': implied_volatility
        }

        # Add the option data to the options_data dictionary
        if quote_date not in options_data:
            options_data[quote_date] = {}
        if underlying not in options_data[quote_date]:
            options_data[quote_date][underlying] = []
        options_data[quote_date][underlying].append(option)

    # Define the VIX parameters
    r = 0.02  # Risk-free interest rate
    K = 30  # Arbitrary value for the VIX calculation

    # Calculate the VIX index for the current date
    vix_index = compute_vix_index(prices, K, options_data, current_date, target_stock)

    # Append the VIX index to the list
    vix_index_list.append((current_date, vix_index))

    # Move to the next date
    start_date += timedelta(days=1)

# Create a DataFrame from the VIX data
vix_df = pd.DataFrame(vix_index_list, columns=['Date', 'VIX Index'])
print(vix_df)
